In [0]:

# Load the cleaned Silver dataset

silver_df = spark.table("workspace.default.silver_secom_quality")

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))


Rows: 1567
Columns: 564


In [0]:

# Convert quality labels into binary classification labels

from pyspark.sql import functions as F

model_df = silver_df.withColumn(
    "label",
    F.when(F.col("quality_label") == 1, 1).otherwise(0)
)

display(
    model_df.groupBy("label")
    .count()
    .orderBy("label")
)


label,count
0,1463
1,104


In [0]:

# Import libraries for classification and evaluation

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Machine-learning libraries imported successfully.")


Machine-learning libraries imported successfully.


In [0]:

# Prepare the feature matrix and target

feature_columns = [
    c for c in model_df.columns
    if c not in ["row_id", "quality_label", "label"]
]

data_pd = model_df.select(
    *feature_columns, "label"
).toPandas()

X = data_pd[feature_columns]
y = data_pd["label"]

print("Records:", len(data_pd))
print("Input features:", len(feature_columns))
print("Pass records:", int((y == 0).sum()))
print("Fail records:", int((y == 1).sum()))


Records: 1567
Input features: 562
Pass records: 1463
Fail records: 104


In [0]:

# Create stratified training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training failures:", int((y_train == 1).sum()))
print("Testing failures:", int((y_test == 1).sum()))


Training records: 1253
Testing records: 314
Training failures: 83
Testing failures: 21


In [0]:

# Train a Random Forest defect classifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest training completed.")


Random Forest training completed.


In [0]:

# Evaluate the model on unseen test records

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Pass", "Fail"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))


Classification Report:
              precision    recall  f1-score   support

        Pass       0.93      1.00      0.97       293
        Fail       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

Confusion Matrix:
[[293   0]
 [ 21   0]]
ROC-AUC: 0.7689


In [0]:

# Split training data into training and validation sets

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Model-training records:", len(X_fit))
print("Validation records:", len(X_val))
print("Validation failures:", int((y_val == 1).sum()))


Model-training records: 1002
Validation records: 251
Validation failures: 17


In [0]:

# Train an improved Random Forest model

improved_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

improved_model.fit(X_fit, y_fit)

print("Improved model training completed.")


Improved model training completed.


In [0]:

# Evaluate failure probabilities on validation data

from sklearn.metrics import precision_recall_curve

val_prob = improved_model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val, val_prob
)

print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_prob), 4))
print("Validation actual failures:", int((y_val == 1).sum()))
print("Validation probabilities calculated:", len(val_prob))


Validation ROC-AUC: 0.7207
Validation actual failures: 17
Validation probabilities calculated: 251


In [0]:

# Compare thresholds using validation data only

from sklearn.metrics import precision_score, recall_score

for threshold in [0.20, 0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(int)

    print(
        "Threshold:", threshold,
        "| Precision:", round(precision_score(y_val, val_pred, zero_division=0), 3),
        "| Recall:", round(recall_score(y_val, val_pred, zero_division=0), 3),
        "| Predicted failures:", int(val_pred.sum())
    )


Threshold: 0.2 | Precision: 0.0 | Recall: 0.0 | Predicted failures: 3
Threshold: 0.3 | Precision: 0.0 | Recall: 0.0 | Predicted failures: 0
Threshold: 0.4 | Precision: 0.0 | Recall: 0.0 | Predicted failures: 0
Threshold: 0.5 | Precision: 0.0 | Recall: 0.0 | Predicted failures: 0


In [0]:

# Inspect the probability range for actual passes and failures

print("Pass probability scores:")
print(pd.Series(val_prob[y_val.to_numpy() == 0]).describe())

print("\nFail probability scores:")
print(pd.Series(val_prob[y_val.to_numpy() == 1]).describe())


Pass probability scores:
count    234.000000
mean       0.073644
std        0.037337
min        0.018827
25%        0.048052
50%        0.067669
75%        0.090849
max        0.245493
dtype: float64

Fail probability scores:
count    17.000000
mean      0.101385
std       0.038994
min       0.028088
25%       0.089667
50%       0.106495
75%       0.132347
max       0.157947
dtype: float64


In [0]:

# Train a balanced Logistic Regression model

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

logistic_model.fit(X_fit, y_fit)

print("Logistic Regression training completed.")


Logistic Regression training completed.


In [0]:

# Compare Logistic Regression with Random Forest

log_val_prob = logistic_model.predict_proba(X_val)[:, 1]
log_val_pred = (log_val_prob >= 0.5).astype(int)

print(
    "Logistic Regression ROC-AUC:",
    round(roc_auc_score(y_val, log_val_prob), 4)
)
print("\nValidation classification report:")
print(classification_report(
    y_val,
    log_val_pred,
    labels=[0, 1],
    target_names=["Pass", "Fail"],
    zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_val, log_val_pred, labels=[0, 1]))


Logistic Regression ROC-AUC: 0.6264

Validation classification report:
              precision    recall  f1-score   support

        Pass       0.94      0.89      0.91       234
        Fail       0.11      0.18      0.13        17

    accuracy                           0.84       251
   macro avg       0.52      0.53      0.52       251
weighted avg       0.88      0.84      0.86       251

Confusion matrix:
[[209  25]
 [ 14   3]]


In [0]:

# Compare Logistic Regression thresholds on validation data

for threshold in [0.20, 0.30, 0.40, 0.50]:
    pred = (log_val_prob >= threshold).astype(int)

    print(
        "Threshold:", threshold,
        "| Precision:", round(precision_score(y_val, pred, zero_division=0), 3),
        "| Recall:", round(recall_score(y_val, pred, zero_division=0), 3),
        "| Predicted failures:", int(pred.sum())
    )


Threshold: 0.2 | Precision: 0.098 | Recall: 0.235 | Predicted failures: 41
Threshold: 0.3 | Precision: 0.111 | Recall: 0.235 | Predicted failures: 36
Threshold: 0.4 | Precision: 0.094 | Recall: 0.176 | Predicted failures: 32
Threshold: 0.5 | Precision: 0.107 | Recall: 0.176 | Predicted failures: 28


In [0]:

# Final test evaluation using Logistic Regression

test_prob = logistic_model.predict_proba(X_test)[:, 1]
test_threshold = 0.30
test_pred = (test_prob >= test_threshold).astype(int)

print("Test threshold:", test_threshold)
print("\nClassification report:")
print(classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    target_names=["Pass", "Fail"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))


Test threshold: 0.3

Classification report:
              precision    recall  f1-score   support

        Pass       0.94      0.82      0.88       293
        Fail       0.10      0.29      0.15        21

    accuracy                           0.79       314
   macro avg       0.52      0.55      0.51       314
weighted avg       0.89      0.79      0.83       314

Confusion matrix:
[[241  52]
 [ 15   6]]
Test ROC-AUC: 0.6317


In [0]:

# Log model evaluation metrics with MLflow

import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

with mlflow.start_run(run_name="SECOM_LogisticRegression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("classification_threshold", test_threshold)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, test_pred))
    mlflow.log_metric(
        "test_precision",
        precision_score(y_test, test_pred, zero_division=0)
    )
    mlflow.log_metric(
        "test_recall",
        recall_score(y_test, test_pred, zero_division=0)
    )
    mlflow.log_metric(
        "test_f1",
        f1_score(y_test, test_pred, zero_division=0)
    )
    mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, test_prob))
    from mlflow.models import infer_signature

    input_example = X_train.head(5)
    output_example = logistic_model.predict(input_example)

    signature = infer_signature(
        input_example,
        output_example
    )

    mlflow.sklearn.log_model(
        sk_model=logistic_model,
        name="secom_logistic_regression",
        input_example=input_example,
        signature=signature
    )

    print("MLflow metrics and model logged successfully.")


🔗 View Logged Model at: https://dbc-03cf7607-d27d.cloud.databricks.com/ml/experiments/4257215871576851/models/m-a105d092468b424aaebf1ba60917a28a?o=7474657567452503


MLflow metrics and model logged successfully.


In [0]:

# Verify the latest MLflow experiment run

run = mlflow.last_active_run()

if run is not None:
    print("Run ID:", run.info.run_id)
    print("Run status:", run.info.status)
    print("Logged metrics:", run.data.metrics)
    print("Logged parameters:", run.data.params)
else:
    print("No active run found in this notebook session.")


Run ID: 586a3b06a59241b1aeac5072583448d2
Run status: FINISHED
Logged metrics: {'test_accuracy': 0.7866242038216561, 'test_f1': 0.1518987341772152, 'test_precision': 0.10344827586206896, 'test_recall': 0.2857142857142857, 'test_roc_auc': 0.6317243620997887}
Logged parameters: {'class_weight': 'balanced', 'classification_threshold': '0.3', 'model_type': 'LogisticRegression'}


In [0]:

# Reload the cleaned dataset and prepare the features

from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split

silver_df = spark.table("workspace.default.silver_secom_quality")

feature_columns = [
    c for c in silver_df.columns
    if c not in ["row_id", "quality_label", "label"]
]

model_df = silver_df.withColumn(
    "label",
    F.when(F.col("quality_label") == 1, 1).otherwise(0)
)

data_pd = model_df.select(
    *feature_columns, "label"
).toPandas()

X = data_pd[feature_columns]
y = data_pd["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Features:", len(feature_columns))
print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Test failures:", int(y_test.sum()))


Features: 562
Training records: 1253
Test records: 314
Test failures: 21


In [0]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Rebuild the model using the same training split
logistic_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression model retrained successfully.")


Logistic Regression model retrained successfully.


In [0]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predict failure probabilities on the held-out test set
test_prob = logistic_model.predict_proba(X_test)[:, 1]

# Use the threshold selected earlier
test_threshold = 0.30
test_pred = (test_prob >= test_threshold).astype(int)

print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Test Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Test Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("Test F1 Score:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))

print("Test predictions generated:", len(test_pred))


Test Accuracy: 0.7994
Test Precision: 0.1613
Test Recall: 0.4762
Test F1 Score: 0.241
Test ROC-AUC: 0.7058
Test predictions generated: 314


In [0]:

import pandas as pd

# Create a results table for the held-out test samples
gold_predictions = pd.DataFrame({
    "test_record_index": X_test.index,
    "actual_label": y_test.to_numpy(),
    "predicted_label": test_pred,
    "failure_probability": test_prob,
    "confidence_score": [
        max(p, 1 - p) for p in test_prob
    ]
})

gold_predictions["prediction_result"] = gold_predictions[
    "predicted_label"
].map({0: "Pass", 1: "Fail"})

gold_predictions["actual_result"] = gold_predictions[
    "actual_label"
].map({0: "Pass", 1: "Fail"})

print("Gold prediction rows:", len(gold_predictions))
print(gold_predictions.head(10).to_string(index=False))


Gold prediction rows: 314
 test_record_index  actual_label  predicted_label  failure_probability  confidence_score prediction_result actual_result
               222             0                0         2.589605e-02          0.974104              Pass          Pass
              1315             0                1         9.996891e-01          0.999689              Fail          Pass
               428             0                0         1.469293e-05          0.999985              Pass          Pass
               277             0                1         7.802475e-01          0.780248              Fail          Pass
              1295             0                0         1.965783e-03          0.998034              Pass          Pass
              1239             0                0         1.239643e-08          1.000000              Pass          Pass
              1432             0                1         9.793875e-01          0.979388              Fail          Pass
      

In [0]:

# Save test predictions to a Gold Delta table

gold_spark_df = spark.createDataFrame(gold_predictions)

gold_table_name = "workspace.default.gold_secom_defect_predictions"

(
    gold_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_name)
)

print("Gold table saved:", gold_table_name)
print("Rows saved:", spark.table(gold_table_name).count())


Gold table saved: workspace.default.gold_secom_defect_predictions
Rows saved: 314


In [0]:

# Verify the saved Gold prediction table

gold_df = spark.table(
    "workspace.default.gold_secom_defect_predictions"
)

gold_df.groupBy(
    "actual_result", "prediction_result"
).count().show()

gold_df.select(
    "test_record_index",
    "actual_result",
    "prediction_result",
    "failure_probability",
    "confidence_score"
).show(10, truncate=False)


+-------------+-----------------+-----+
|actual_result|prediction_result|count|
+-------------+-----------------+-----+
|         Pass|             Pass|  241|
|         Pass|             Fail|   52|
|         Fail|             Pass|   11|
|         Fail|             Fail|   10|
+-------------+-----------------+-----+

+-----------------+-------------+-----------------+---------------------+------------------+
|test_record_index|actual_result|prediction_result|failure_probability  |confidence_score  |
+-----------------+-------------+-----------------+---------------------+------------------+
|222              |Pass         |Pass             |0.02589604728982944  |0.9741039527101706|
|1315             |Pass         |Fail             |0.9996890855826704   |0.9996890855826704|
|428              |Pass         |Pass             |1.469292504713292E-5 |0.9999853070749528|
|277              |Pass         |Fail             |0.780247507221345    |0.780247507221345 |
|1295             |Pass     

In [0]:

import mlflow
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

with mlflow.start_run(run_name="SECOM_LogisticRegression_Retrained"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("classification_threshold", 0.30)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, test_pred, zero_division=0))
    mlflow.log_metric("test_recall", recall_score(y_test, test_pred, zero_division=0))
    mlflow.log_metric("test_f1", f1_score(y_test, test_pred, zero_division=0))
    mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, test_prob))

print("Latest model metrics logged to MLflow.")


Latest model metrics logged to MLflow.


In [0]:

# Verify the most recent MLflow run

runs = mlflow.search_runs(
    order_by=["start_time DESC"],
    max_results=3
)

display(
    runs[[
        "run_id",
        "status",
        "tags.mlflow.runName",
        "metrics.test_accuracy",
        "metrics.test_recall",
        "metrics.test_f1",
        "metrics.test_roc_auc"
    ]]
)


run_id,status,tags.mlflow.runName,metrics.test_accuracy,metrics.test_recall,metrics.test_f1,metrics.test_roc_auc
4df587e4a8c64d119754e686ca41e01b,FINISHED,SECOM_LogisticRegression_Retrained,0.7993630573248408,0.47619047619047616,0.24096385542168675,0.7058345522509345
586a3b06a59241b1aeac5072583448d2,FINISHED,SECOM_LogisticRegression,0.7866242038216561,0.2857142857142857,0.1518987341772152,0.6317243620997887
7328dd88deaf4fdb8c023f994ebdbeaa,FINISHED,SECOM_LogisticRegression_Retrained,0.7993630573248408,0.47619047619047616,0.24096385542168675,0.7058345522509345


In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

model_name = "workspace.default.secom_defect_classifier"

print("Ready to register:", model_name)

Ready to register: workspace.default.secom_defect_classifier


In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name("/Shared/03_ML_Model")

if experiment is None:
    print("Experiment not found by this name.")
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=10
    )

    for run in runs:
        print(
            "Run ID:", run.info.run_id,
            "| Run name:", run.data.tags.get("mlflow.runName"),
            "| Status:", run.info.status
        )

Experiment not found by this name.


In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

runs = client.search_runs(
    experiment_ids=["4257215871576851"],
    order_by=["start_time DESC"],
    max_results=10
)

for run in runs:
    print(
        "Run ID:", run.info.run_id,
        "| Run name:", run.data.tags.get("mlflow.runName"),
        "| Status:", run.info.status
    )

Run ID: 4df587e4a8c64d119754e686ca41e01b | Run name: SECOM_LogisticRegression_Retrained | Status: FINISHED
Run ID: 586a3b06a59241b1aeac5072583448d2 | Run name: SECOM_LogisticRegression | Status: FINISHED
Run ID: 7328dd88deaf4fdb8c023f994ebdbeaa | Run name: SECOM_LogisticRegression_Retrained | Status: FINISHED
Run ID: fd780b934de149a9a6f8f7c0c67b44b0 | Run name: SECOM_LogisticRegression | Status: FINISHED
Run ID: 95e078822dd5458aa952fda55b4e2e96 | Run name: SECOM_LogisticRegression | Status: FINISHED
Run ID: 3a4c0951c5e249359f1ab19dcdf0c366 | Run name: SECOM_LogisticRegression | Status: FINISHED
Run ID: 0eca54a410fb46809b40cbda2e382dad | Run name: SECOM_LogisticRegression | Status: FINISHED
Run ID: 4d5968b089f643e8b51019619f4d82c8 | Run name: SECOM_LogisticRegression | Status: FAILED
Run ID: 3ad76c536d794fa49e6171be232d9db0 | Run name: SECOM_LogisticRegression_Retrained | Status: FINISHED
Run ID: b6ed04b8c6bc486b8cc58ab0e8abb0f2 | Run name: SECOM_LogisticRegression | Status: FINISHED


In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

run_id = "3ad76c536d794fa49e6171be232d9db0"

run = client.get_run(run_id)

print("Run:", run.info.run_id)
print("Artifacts:")
for item in client.list_artifacts(run_id):
    print(item.path, "| directory:", item.is_dir)

Run: 3ad76c536d794fa49e6171be232d9db0
Artifacts:


In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

experiment_id = "4257215871576851"

runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["start_time DESC"],
    max_results=10
)

for run in runs:
    print(
        "Run ID:", run.info.run_id,
        "| Name:", run.data.tags.get("mlflow.runName"),
        "| Artifacts URI:", run.info.artifact_uri
    )

Run ID: 4df587e4a8c64d119754e686ca41e01b | Name: SECOM_LogisticRegression_Retrained | Artifacts URI: dbfs:/databricks/mlflow-tracking/4257215871576851/4df587e4a8c64d119754e686ca41e01b/artifacts
Run ID: 586a3b06a59241b1aeac5072583448d2 | Name: SECOM_LogisticRegression | Artifacts URI: dbfs:/databricks/mlflow-tracking/4257215871576851/586a3b06a59241b1aeac5072583448d2/artifacts
Run ID: 7328dd88deaf4fdb8c023f994ebdbeaa | Name: SECOM_LogisticRegression_Retrained | Artifacts URI: dbfs:/databricks/mlflow-tracking/4257215871576851/7328dd88deaf4fdb8c023f994ebdbeaa/artifacts
Run ID: fd780b934de149a9a6f8f7c0c67b44b0 | Name: SECOM_LogisticRegression | Artifacts URI: dbfs:/databricks/mlflow-tracking/4257215871576851/fd780b934de149a9a6f8f7c0c67b44b0/artifacts
Run ID: 95e078822dd5458aa952fda55b4e2e96 | Name: SECOM_LogisticRegression | Artifacts URI: dbfs:/databricks/mlflow-tracking/4257215871576851/95e078822dd5458aa952fda55b4e2e96/artifacts
Run ID: 3a4c0951c5e249359f1ab19dcdf0c366 | Name: SECOM_Logis

In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

for run in client.search_runs(
    experiment_ids=["4257215871576851"],
    order_by=["start_time DESC"],
    max_results=15
):
    print(
        run.info.run_id,
        "|",
        run.data.tags.get("mlflow.runName"),
        "|",
        run.info.status
    )
    for artifact in client.list_artifacts(run.info.run_id):
        print("   Artifact:", artifact.path)

4df587e4a8c64d119754e686ca41e01b | SECOM_LogisticRegression_Retrained | FINISHED
586a3b06a59241b1aeac5072583448d2 | SECOM_LogisticRegression | FINISHED
7328dd88deaf4fdb8c023f994ebdbeaa | SECOM_LogisticRegression_Retrained | FINISHED
fd780b934de149a9a6f8f7c0c67b44b0 | SECOM_LogisticRegression | FINISHED
95e078822dd5458aa952fda55b4e2e96 | SECOM_LogisticRegression | FINISHED
3a4c0951c5e249359f1ab19dcdf0c366 | SECOM_LogisticRegression | FINISHED
0eca54a410fb46809b40cbda2e382dad | SECOM_LogisticRegression | FINISHED
4d5968b089f643e8b51019619f4d82c8 | SECOM_LogisticRegression | FAILED
3ad76c536d794fa49e6171be232d9db0 | SECOM_LogisticRegression_Retrained | FINISHED
b6ed04b8c6bc486b8cc58ab0e8abb0f2 | SECOM_LogisticRegression | FINISHED


In [0]:
try:
    from databricks.feature_engineering import FeatureEngineeringClient
    print("Feature Engineering client is available.")
except ImportError as e:
    print("Feature Engineering client is not available:", e)

Feature Engineering client is not available: No module named 'databricks.feature_engineering'


In [0]:
%pip install databricks-feature-engineering

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.3 MB/s  0:00:00
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.122.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1f5dba2a-f823-4d25-80d2-9a27e2e48fa4
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [databricks-feature-engineering]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

print("Feature Engineering client is ready!")

Feature Engineering client is ready!


In [0]:
silver_df = spark.table("workspace.default.silver_secom_quality")

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))
print("First 10 columns:", silver_df.columns[:10])

Rows: 1567
Columns: 564
First 10 columns: ['row_id', 'feature_001', 'feature_002', 'feature_003', 'feature_004', 'feature_005', 'feature_006', 'feature_007', 'feature_008', 'feature_009']


In [0]:
feature_columns = [
    c for c in silver_df.columns
    if c not in ["row_id", "quality_label"]
]

feature_df = silver_df.select("row_id", *feature_columns)

print("Rows:", feature_df.count())
print("Feature columns:", len(feature_columns))
print("Total columns including key:", len(feature_df.columns))

Rows: 1567
Feature columns: 562
Total columns including key: 563


In [0]:
feature_table = fe.create_table(
    name="workspace.default.secom_defect_features",
    primary_keys=["row_id"],
    df=feature_df,
    description="Reusable SECOM sensor features for manufacturing defect classification"
)

print("Feature table created successfully!")

2026/10/28 15:48:20 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['row_id'] of table 'workspace.default.secom_defect_features' to NOT NULL.
2026/10/28 15:48:23 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['row_id'] on table 'workspace.default.secom_defect_features'.
2026/10/28 15:48:34 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.default.secom_defect_features'.


Feature table created successfully!


In [0]:
fe_table_df = spark.table("workspace.default.secom_defect_features")

print("Rows:", fe_table_df.count())
print("Columns:", len(fe_table_df.columns))
fe_table_df.select("row_id").show(5)

Rows: 1567
Columns: 563
+------+
|row_id|
+------+
|     1|
|     2|
|     3|
|     4|
|     5|
+------+
only showing top 5 rows


In [0]:
print("Spark version:", spark.version)
print("Python version:", __import__("sys").version)

Spark version: 4.2.0
Python version: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


In [0]:
import sklearn
import mlflow

print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)
print("Ready to compare classification models.")

scikit-learn: 1.7.2
MLflow: 3.12.0
Ready to compare classification models.


In [0]:

import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# Load the existing cleaned data
silver_df = spark.table("workspace.default.silver_secom_quality")

feature_columns = [
    c for c in silver_df.columns
    if c not in ["row_id", "quality_label", "label"]
]

model_df = silver_df.withColumn(
    "label",
    __import__("pyspark").sql.functions.when(
        __import__("pyspark").sql.functions.col("quality_label") == 1, 1
    ).otherwise(0)
)

data_pd = model_df.select(*feature_columns, "label").toPandas()
X = data_pd[feature_columns]
y = data_pd["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "LogisticRegression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    ),
    "DecisionTree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            random_state=42
        )
    ),
    "RandomForest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
}

mlflow.set_experiment("/Shared/SECOM_Model_Comparison")
results = []

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(
                y_test, pred, zero_division=0
            ),
            "recall": recall_score(
                y_test, pred, zero_division=0
            ),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, prob)
        }

        mlflow.log_param("model_type", name)
        mlflow.log_param("positive_class", "Fail")
        mlflow.log_metrics(metrics)

        results.append({"model": name, **metrics})

comparison_df = pd.DataFrame(results).sort_values(
    "f1", ascending=False
)

display(comparison_df)
print("Comparison runs logged to MLflow.")


2026/10/28 15:54:22 INFO mlflow.tracking.fluent: Experiment with name '/Shared/SECOM_Model_Comparison' does not exist. Creating a new experiment.


model,accuracy,precision,recall,f1,roc_auc
LogisticRegression,0.821656050955414,0.16981132075471697,0.42857142857142855,0.24324324324324326,0.7058345522509345
DecisionTree,0.7643312101910829,0.136986301369863,0.47619047619047616,0.2127659574468085,0.5856492767755566
RandomForest,0.9331210191082803,0.0,0.0,0.0,0.7792946530147894


Comparison runs logged to MLflow.
